# 📈 GRU Demand Forecasting (Daily Aggregation)

Model prediksi permintaan (Order Item Quantity) **harian** menggunakan **Gated Recurrent Unit (GRU)** dengan PyTorch.
Telah dioptimasi untuk mencegah overfitting dengan menggunakan *daily aggregation*, arsitektur model yang lebih kecil, *L2 Regularization*, dan *Early Stopping* yang lebih ketat.

Fitur yang digunakan:
- `Order Item Quantity` (Target)
- `Sales`
- `Delay`
- `Product Price`
- `RouteRisk`
- `TempDev` (Temperature Deviation)
- `Shipping Mode`
- `Market`

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 2. Load Data & Sort by Date

In [ ]:
path = "./data/cold_chain_supply_chain.csv"
df = pd.read_csv(path)

# Parse dates and sort
df['order_date'] = pd.to_datetime(df['order date (DateOrders)'], format='mixed')
df = df.sort_values('order_date')
df.set_index('order_date', inplace=True)

print(f"Data loaded: {df.shape[0]} rows, from {df.index.min()} to {df.index.max()}")

## 3. Aggregate Time Series (Daily)
Agregasi data menjadi **harian** untuk meminimalkan kehilangan variasi pola dan meningkatkan jumlah sequence (1000+ observasi).
Tanggal dengan nilai kosong akan di-*fill* (0 untuk kuantitas, ffill untuk sisanya).

In [ ]:
daily = df.resample('D').agg({
    'Order Item Quantity': 'sum',
    'Sales': 'sum',
    'Delay': 'mean',
    'Product Price': 'mean',
    'RouteRisk': 'mean',
    'TempDev': 'mean',
    'Shipping Mode': lambda x: x.mode()[0] if not x.empty else np.nan,
    'Market': lambda x: x.mode()[0] if not x.empty else np.nan
})

# Penanganan hari kosong (missing days)
daily['Order Item Quantity'] = daily['Order Item Quantity'].fillna(0)
daily['Sales'] = daily['Sales'].fillna(0)

# Forward-fill untuk fitur rata-rata dan kategorikal
daily = daily.ffill().bfill()

print(f"Daily aggregated shape: {daily.shape}")
daily.head()

## 4. Normalization & Feature Encoding
One-hot encoding untuk fitur kategorikal dan `MinMaxScaler` untuk normalisasi ke skala 0-1 untuk input GRU.

In [ ]:
# One-hot encoding untuk fitur kategorikal
daily_encoded = pd.get_dummies(daily, columns=['Shipping Mode', 'Market'], drop_first=True)

# Pisahkan Train / Test (80% / 20% secara kronologis)
train_size = int(len(daily_encoded) * 0.8)
train_df = daily_encoded.iloc[:train_size]
test_df = daily_encoded.iloc[train_size:]

# Simpan nama kolom untuk indeks
features = daily_encoded.columns.tolist()
target_col = 'Order Item Quantity'
target_idx = features.index(target_col)

# Normalisasi menggunakan MinMaxScaler (fit hanya pada train_df untuk menghindari data leakage)
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_df)
test_scaled = scaler.transform(test_df)

# Scaler khusus target untuk inverse_transform saat evaluasi nanti
target_scaler = MinMaxScaler()
target_scaler.fit(train_df[[target_col]])

print(f"Train size: {len(train_df)} days, Test size: {len(test_df)} days")

## 5. Create Sequences
Membuat *sliding window* dengan **14 timestep (hari)** sebagai input untuk memprediksi 1 hari berikutnya.

In [ ]:
def create_sequences(data, target_col_idx, window=14):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i-window:i])
        y.append(data[i, target_col_idx])
    return np.array(X), np.array(y)

window_size = 14

X_train, y_train = create_sequences(train_scaled, target_idx, window_size)
X_test, y_test = create_sequences(test_scaled, target_idx, window_size)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

## 6. Build GRU Model
Arsitektur dikecilkan untuk mencegah overfitting: `GRU(16) -> Dropout -> Dense(1)`

In [ ]:
# Konversi ke PyTorch Dataset
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(TimeSeriesDataset(X_train, y_train), batch_size=32, shuffle=False)
test_loader = DataLoader(TimeSeriesDataset(X_test, y_test), batch_size=32, shuffle=False)

# Definisi Model GRU
class GRUNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=16, output_dim=1, dropout=0.2):
        super(GRUNet, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        out, _ = self.gru(x)
        out = out[:, -1, :]  # Ambil output timestep terakhir
        out = self.dropout(out)
        out = self.fc(out)
        return out

input_dim = X_train.shape[2]
model = GRUNet(input_dim=input_dim, hidden_dim=16, dropout=0.2)
print(model)

## 7. Train Model
Melatih model dengan fungsi Loss MSE, optimizer Adam (*weight decay* `1e-5` untuk L2 Regularization), dan *Early Stopping* ketat (`patience = 5`).

In [ ]:
criterion = nn.MSELoss()
# Tambahkan L2 Regularization dengan weight_decay
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

epochs = 150
train_losses, val_losses = [], []

best_val_loss = float('inf')
patience, patience_counter = 5, 0  # Pengetatan early stopping
best_model_state = None

for epoch in range(epochs):
    model.train()
    batch_losses = []
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())
        
    train_loss = np.mean(batch_losses)
    train_losses.append(train_loss)
    
    model.eval()
    val_batch_losses = []
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            outputs = model(batch_X)
            val_loss = criterion(outputs, batch_y)
            val_batch_losses.append(val_loss.item())
            
    val_loss = np.mean(val_batch_losses)
    val_losses.append(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        patience_counter = 0
    else:
        patience_counter += 1
        
    if (epoch + 1) % 10 == 0 or patience_counter >= patience:
        print(f"Epoch {epoch+1:03d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Restore best weights
if best_model_state is not None:
    model.load_state_dict(best_model_state)

# Plot Learning Curve
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Learning Curves (MSE)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

## 8. Evaluation
Fokus evaluasi pada **MAE** (rata-rata kesalahan prediksi harian) dan **RMSE** (sensitif terhadap outlier).
Metrik MAPE dapat meroket karena target kuantitas harian bisa memiliki nilai yang mendekati nol.

In [ ]:
model.eval()
predictions, actuals = [], []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        outputs = model(batch_X)
        predictions.extend(outputs.numpy())
        actuals.extend(batch_y.numpy())

predictions = np.array(predictions)
actuals = np.array(actuals)

# Inverse transform untuk mengembalikan nilai ke satuan asli (Quantity)
predictions_inv = target_scaler.inverse_transform(predictions)
actuals_inv = target_scaler.inverse_transform(actuals)

# Hitung Metrics
mae = mean_absolute_error(actuals_inv, predictions_inv)
rmse = np.sqrt(mean_squared_error(actuals_inv, predictions_inv))
mape = mean_absolute_percentage_error(actuals_inv, predictions_inv)

print(f"=== EVALUATION METRICS ===")
print(f"MAE  : {mae:.2f} items/day")
print(f"RMSE : {rmse:.2f} items/day")
print(f"MAPE : {mape * 100:.2f} %")

# Plot Actual vs Predicted
plt.figure(figsize=(14, 6))
test_dates = test_df.index[window_size:]

plt.plot(test_dates, actuals_inv, label='Actual Quantity', alpha=0.7, color='#2ecc71')
plt.plot(test_dates, predictions_inv, label='Predicted Quantity', alpha=0.9, color='#e74c3c')
plt.title('GRU Demand Forecasting: Actual vs Predicted (Daily)')
plt.xlabel('Date')
plt.ylabel('Order Item Quantity')
plt.legend()
plt.show()